# 17段を含む分類基準の感度分析
T1〜T5を順に実行します。既存モデルAの20特徴量を使い、baseline・include_17_low・include_17_highを比較します。主分析は非重複10対局、感度分析は20対局です。
既存のC5には17段を含む全対局が記録される設計です。対象集合と分子・分母の署名を照合し、baselineの予測・評価結果が従来と一致するか検査します。
3条件とも同じ時系列境界を使い、クラスが揃う連続対局の窓を作ります。条件によって採用窓が変わるため、同じ元対局で構成された共通評価窓の比較も出力します。
T4は17段を含む2条件のモデルAを固定設定で学習します。最大12モデルです。第3段階のbaselineモデルは保存済みのものを読み込みます。実データの結果は実行後に作成されます。

In [ ]:
# T1　Drive接続・固定版コードの取得・合成データでの検証
from google.colab import drive
from pathlib import Path
import sys, os, subprocess, tempfile, urllib.request, importlib.metadata, importlib.util, json
drive.mount('/content/drive')
PIN = '73d9416e634e1946f9a4fcef4c0d2148f189ab3f'
src = Path(tempfile.mkdtemp(prefix='mahjong_threshold_'))
files = ['mahjong_research/'+f for f in ('__init__.py','analysis.py','inputs.py','tenhou.py','legacy.py')]
files += ['smoothing/study.py','smoothing/presentation.py','smoothing/test_smoothing.py',
          'stability/stability.py','choices/choices.py','relationship_analysis/relationships.py']
files += ['model_comparison/'+f for f in ('windows.py','compare.py','test_comparison.py')]
files += ['rank_threshold_sensitivity/'+f for f in ('thresholds.py','threshold_report.py','test_thresholds.py')]
for rel in files:
    target=src/rel
    target.parent.mkdir(exist_ok=True)
    url=f'https://raw.githubusercontent.com/yuukienomoto/report_-syobon_n/{PIN}/research_v2/{rel}'
    with urllib.request.urlopen(url,timeout=60) as response:
        target.write_bytes(response.read())
needed=[pkg for module,pkg in [('numpy','numpy'),('pandas','pandas'),('matplotlib','matplotlib'),('sklearn','scikit-learn')]
        if importlib.util.find_spec(module) is None]
if importlib.util.find_spec('lightgbm') is None or importlib.metadata.version('lightgbm')!='4.6.0':
    needed.append('lightgbm==4.6.0')
if needed:
    subprocess.run([sys.executable,'-m','pip','install','-q',*needed],check=True)
paths=[str(src/p) for p in ('rank_threshold_sensitivity','relationship_analysis','model_comparison','stability','smoothing','choices','.')]
env=dict(os.environ,PYTHONPATH=os.pathsep.join(paths),MPLBACKEND='Agg')
subprocess.run([sys.executable,'-m','unittest','-q','test_thresholds'],cwd=src,env=env,check=True)

In [ ]:
# T2　保存済み入力と今回の出力先。通常は既定値を使用できます
ROOT=Path('/content/drive/MyDrive/卒論')
RUN=ROOT/'research_v2_runs/20261105T071722Z_5da6c9ad'
PHASE1=ROOT/'research_v2_stability/20261105T092459Z_113b116b'
PHASE3=ROOT/'research_v2_model_comparison/20261117T160007Z_113bc0ca'
OUTPUT=ROOT/'research_v2_rank_threshold_sensitivity'
ULTRA_CACHE=None  # 外部C8キャッシュを移動した場合だけ新しいフルパス
RESUME=None  # 中断後はT4が表示した今回の日時付きフォルダを指定
for p in (RUN/'input_plan.json', PHASE1/'fold_membership.csv', PHASE3/'protocol.json'):
    if not p.is_file():
        raise FileNotFoundError(f'入力が見つかりません。T2のパスを確認してください {p}')
print('出力の親',OUTPUT)

In [ ]:
# T3　学習前の入力監査。段位別対局数・窓数・共通の分割境界を確認
args=[str(p) for p in (RUN,PHASE1,PHASE3)]
audit_code="""import sys,json
from thresholds import prepare
info=prepare(*json.loads(sys.argv[1]),ultra_cache=json.loads(sys.argv[2]))
for name,data in [('しょぼんｎ',info['data']),('ウルトラ立直',info['external_raw'])]:
    if data is not None:
        print(name,'キャッシュの段位別対局数')
        print(data.dan.value_counts().sort_index().to_string())
for condition,audit in info['audits'].items():
    print(condition)
    print(audit[['width','fold','train_through','eval_through','train_low','train_high','eval_low','eval_high','dropped_crossing','status']].to_string(index=False))
print(info['external_note'])
"""
subprocess.run([sys.executable,'-u','-c',audit_code,json.dumps(args),json.dumps(str(ULTRA_CACHE) if ULTRA_CACHE else None)],
               cwd=src,env=env,check=True)

In [ ]:
# T4　baseline再現確認・固定条件での追加学習・外部評価・レポート保存
args=[str(p) for p in (RUN,PHASE1,PHASE3,OUTPUT)]
args += [str(RESUME) if RESUME else None,str(ULTRA_CACHE) if ULTRA_CACHE else None]
code="""import sys,json
from pathlib import Path
from thresholds import run
out=run(*json.loads(sys.argv[1]))
Path(sys.argv[2]).write_text(str(out),encoding='utf8')
"""
result_path=src/'result_path.txt'
subprocess.run([sys.executable,'-u','-c',code,json.dumps(args),str(result_path)],cwd=src,env=env,check=True)
RESULT=Path(result_path.read_text(encoding='utf8'))
print('保存先',RESULT)

In [ ]:
# T5　比較結果と解釈上の制約を表示
import pandas as pd
from IPython.display import display,Markdown,Image
display(pd.read_csv(RESULT/'baseline_reproduction.csv'))
display(pd.read_csv(RESULT/'metrics_comparison.csv'))
display(pd.read_csv(RESULT/'performance_changes.csv'))
display(Image(filename=str(RESULT/'auc_comparison.png')))
display(Markdown((RESULT/'report_ja.md').read_text(encoding='utf8')))

保存先は新しい日時付きフォルダです。中断時はT4が表示したフォルダをT2のRESUMEに指定し、T3・T4を再実行すると、署名が一致する学習済みモデルを再利用します。
baselineの不一致は入力・特徴量・保存モデルを確認するために停止します。17段の行が不足している場合は、エラーに示されたC5キャッシュとinput_plan.jsonの抽出対象を確認してください。外部キャッシュが利用できない場合は理由を出力し、内部比較を完了します。
order_verified=False、full_history_confirmed=Falseを維持します。段位17は天鳳牌譜のコード上の区分を表します。既存結果を確認後の探索的分析であり、最も高いAUCの条件が独立に検証された最適基準とはいえません。